## Telegram crawler

### Setup

- Install deps: `pip install telethon nest_asyncio`.
- Create a `.env` in the repo root with `TG_API_ID`, `TG_API_HASH`, `TG_SESSION_NAME`  where to store the credential from your Telegram account
- Run the notebook cells in order; the crawl writes JSON/CSVs per seed into `data/<seed>/`.

In [4]:
startSeed = "generationidentitaire" #forzanuovafn #mihazank  #afdjugendbw

maxDepth = 2, # how many hops from seed channels
maxPerChat = 300, # max number of subs to fetch per channel
handleDelay = 3.0, # delay between handling different channels
maxWait = 600, # max wait time for Telegram responses
maxSubsForMessages = 30000, # max number of subs to fetch messages from
outputPath = "data/graph.json", # path to output graph data

In [5]:
import os
import asyncio
import json
import re
import csv
import pathlib
from collections import deque
from typing import Dict, Iterable, List, Set, Tuple

from telethon import TelegramClient
from telethon.errors import FloodWaitError
from telethon.tl.functions.channels import GetFullChannelRequest
from telethon.tl.functions.messages import GetFullChatRequest
from telethon.tl.types import Chat, Channel, Message, MessageEntityMention, MessageEntityMentionName, MessageEntityTextUrl

In [ ]:
def loadEnvFromFile(path: str = ".env") -> None:
    if not os.path.exists(path):
        return
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            stripped = line.strip()
            if not stripped or stripped.startswith("#") or "=" not in stripped:
                continue
            key, _, value = stripped.partition("=")
            if key and value and key not in os.environ:
                os.environ[key] = value


def loadConfig() -> Dict:
    loadEnvFromFile()
    output_dir_env = os.getenv("TG_OUTPUT_DIR", "").strip()
    output_dir = (output_dir_env or os.path.dirname(outputPath) or "data") or "data"
    config: Dict[str, object] = {
        "apiId": os.getenv("TG_API_ID"),
        "apiHash": os.getenv("TG_API_HASH"),
        "sessionName": os.getenv("TG_SESSION_NAME", "session_name"),
        "startSeeds": [s.strip() for s in startSeed.split(",") if s.strip()],
        "maxDepth": maxDepth,
        "maxPerChat": maxPerChat,
        "outputPath": outputPath,
        "outputDir": output_dir,
        "handleDelay": handleDelay,
        "maxWait": maxWait,
        "maxSubsForMessages": maxSubsForMessages,
    }
    try:
        maxPerChatVal = int(config["maxPerChat"]) 
    except Exception:
        maxPerChatVal = -1
    config["maxPerChat"] = None if maxPerChatVal <= 0 else maxPerChatVal
    if not config["apiId"] or not config["apiHash"]:
        raise RuntimeError("Missing TG_API_ID/TG_API_HASH")
    return config


config = loadConfig()
config


TypeError: expected str, bytes or os.PathLike object, not tuple

In [ ]:
def isLikelyUsername(s: str) -> bool:
    if not s:
        return False
    s = s.strip()
    if len(s) < 5 or len(s) > 32:
        return False
    if not re.fullmatch(r"[A-Za-z][\w\d]{3,30}[A-Za-z\d]", s):
        return False
    return True


def handleFromUrl(url: str) -> str:
    match = re.search(r"(?:https?://)?t\.me/(?:c/\d+/)?([A-Za-z0-9_]+)", url or "")
    return match.group(1) if match else ""


def parseTmeMessageLink(url: str) -> Tuple[str, str]:
    match = re.search(r"(?:https?://)?t\.me/([A-Za-z0-9_]+)/([0-9]+)", url or "")
    return (match.group(1), match.group(2)) if match else ("", "")


def normalize_seed(seed: str) -> str:
    return (seed or "").strip().lower()


def collectReactions(message: Message) -> Tuple[int, Dict[str, int]]:
    counts: Dict[str, int] = {}
    if message.reactions and getattr(message.reactions, 'results', None):
        for reaction in message.reactions.results:
            emoji = getattr(reaction.reaction, 'emoticon', str(reaction.reaction))
            counts[emoji] = counts.get(emoji, 0) + reaction.count
    return sum(counts.values()), counts


def extractMentionsAndLinks(message: Message) -> Tuple[Set[str], List[Tuple[str, str, str]]]:
    chats: Set[str] = set()
    links: List[Tuple[str, str, str]] = []
    if not message:
        return chats, links
    if message.entities:
        text = message.message or ""
        for entity in message.entities:
            if isinstance(entity, MessageEntityMention):
                handle = text[entity.offset: entity.offset + entity.length].lstrip("@")
                if isLikelyUsername(handle):
                    chats.add(handle)
            elif isinstance(entity, MessageEntityMentionName):
                continue
            elif isinstance(entity, MessageEntityTextUrl):
                handle = handleFromUrl(entity.url)
                if isLikelyUsername(handle):
                    chats.add(handle)
                msgHandle, msgId = parseTmeMessageLink(entity.url)
                if isLikelyUsername(msgHandle) and msgId:
                    links.append((msgHandle, msgId, "link"))
    if message.forward and message.forward.chat:
        username = getattr(message.forward.chat, "username", None)
        if username and isLikelyUsername(username):
            chats.add(username)
            fwdMsgId = (
                getattr(message.forward, "channel_post", None)
                or getattr(message.forward, "saved_from_msg_id", None)
                or getattr(message.forward, "msg_id", None)
            )
            if fwdMsgId:
                links.append((username, str(fwdMsgId), "forward"))
    if message.message:
        for match in re.findall(r"(?:https?://)?t\.me/(?:c/\d+/)?([A-Za-z0-9_]+)", message.message):
            if isLikelyUsername(match):
                chats.add(match)
        for match in re.findall(r"(?:https?://)?t\.me/([A-Za-z0-9_]+)/([0-9]+)", message.message):
            if isLikelyUsername(match[0]):
                chats.add(match[0])
                links.append((match[0], match[1], "link"))
    return chats, links


def getTitle(entity, handle: str) -> str:
    return getattr(entity, "title", None) or getattr(entity, "first_name", "") or handle


async def getSubscriberCount(client: TelegramClient, entity) -> int:
    try:
        if isinstance(entity, Channel):
            full = await client(GetFullChannelRequest(entity))
            return int(getattr(full.full_chat, "participants_count", 0) or 0)
        if isinstance(entity, Chat):
            full = await client(GetFullChatRequest(entity.id))
            return int(getattr(full.full_chat, "participants_count", 0) or 0)
    except Exception as exc:
        print(f"[warn] subscribers for {getattr(entity, 'username', entity)} failed: {exc}")
    return 0


async def fetchEntityWithBackoff(client: TelegramClient, handle: str, maxWait: int):
    handle = (handle or "").strip()
    if not isLikelyUsername(handle):
        print(f"[skip] {handle}: looks invalid, skipping before API call")
        return None
    while True:
        try:
            return await client.get_entity(handle)
        except FloodWaitError as exc:
            wait = int(getattr(exc, "seconds", 0) or 0)
            if maxWait and wait > maxWait:
                print(f"[skip] {handle}: flood wait {wait}s exceeds maxWait={maxWait}s; skipping")
                return None
            print(f"[wait] {handle}: flood wait {wait}s on get_entity; sleeping...")
            await asyncio.sleep(wait + 1)
        except Exception as exc:
            print(f"[skip] {handle}: {exc}")
            return None


In [ ]:
async def crawlGraph(
    client: TelegramClient,
    seeds: Iterable[str],
    maxDepth: int,
    maxPerChat,
    handleDelay: float,
    maxWait: int,
    maxSubsForMessages: int,
) -> Dict[str, List[Dict[str, str]]]:
    visited: Set[str] = set()
    queue: deque[Tuple[str, int]] = deque((seed, 0) for seed in seeds)
    nodes: Dict[str, Dict[str, str]] = {}
    edges: List[Dict[str, str]] = []
    messages: List[Dict[str, str]] = []
    messageEdges: List[Dict[str, str]] = []

    while queue:
        handle, depth = queue.popleft()
        if handle in visited or depth > maxDepth:
            continue

        if handleDelay > 0:
            await asyncio.sleep(handleDelay)

        entity = await fetchEntityWithBackoff(client, handle, maxWait)
        if not entity:
            visited.add(handle)
            continue

        title = getTitle(entity, handle)
        subscribers = await getSubscriberCount(client, entity)
        if subscribers < 200:
            print(f"[skip-subs] {handle}: {subscribers} subscribers < 200, skipping node and children")
            visited.add(handle)
            continue

        nodes[handle] = {"id": handle, "title": title, "subscribers": subscribers}
        visited.add(handle)
        print(f"[scrape] {handle} depth={depth}")

        if maxSubsForMessages and subscribers > maxSubsForMessages:
            print(f"[skip-msgs] {handle}: {subscribers} subscribers > {maxSubsForMessages}, skipping messages")
            continue

        retries = 0
        maxRetriesPerChat = 3

        while True:
            try:
                fetchLimit = None if maxPerChat is None else maxPerChat
                async for message in client.iter_messages(entity, limit=fetchLimit):
                    msgKey = f"{handle}:{message.id}"
                    reactionCount, reactionDetails = collectReactions(message)
                    messages.append({
                        "id": msgKey,
                        "chat": handle,
                        "message_id": str(message.id),
                        "date": message.date.isoformat() if message.date else "",
                        "text": message.message or "",
                        "views": int(message.views or 0),
                        "sender_id": message.sender_id,
                        "reaction_count": reactionCount,
                        "reaction_breakdown": reactionDetails,
                        "url": f"https://t.me/{handle}/{message.id}",
                    })

                    chatMentions, msgLinks = extractMentionsAndLinks(message)
                    for target in chatMentions:
                        edges.append({"from": handle, "to": target, "message_id": str(message.id)})
                        if target not in visited and depth < maxDepth:
                            queue.append((target, depth + 1))

                    for targetChat, targetMsgId, reason in msgLinks:
                        messageEdges.append({"from": msgKey, "to": f"{targetChat}:{targetMsgId}", "type": reason})

                    if getattr(message, "reply_to_msg_id", None):
                        messageEdges.append({"from": msgKey, "to": f"{handle}:{message.reply_to_msg_id}", "type": "reply"})
                break
            except FloodWaitError as exc:
                wait = int(getattr(exc, "seconds", 0) or 0)
                retries += 1
                if (maxWait and wait > maxWait) or retries >= maxRetriesPerChat:
                    print(f"[skip] {handle}: flood wait {wait}s on iter_messages, retries={retries}; skipping chat")
                    break
                print(f"[wait] {handle}: flood wait {wait}s on iter_messages; sleeping and resuming...")
                await asyncio.sleep(wait + 1)
                continue
            except Exception as exc:
                print(f"[warn] iter_messages for {handle} failed: {exc}")
                break

    return {"nodes": list(nodes.values()), "edges": edges, "messages": messages, "message_edges": messageEdges}


def write_graph_outputs(graph: Dict[str, List[Dict[str, str]]], output_dir, seed: str = ""):
    output_dir = pathlib.Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    graph_path = output_dir / "graph.json"
    graph_path.write_text(json.dumps(graph, ensure_ascii=False, indent=2), encoding="utf-8")

    def write_csv(path, rows, fieldnames):
        with open(path, "w", newline="", encoding="utf-8") as fp:
            writer = csv.DictWriter(fp, fieldnames=fieldnames)
            writer.writeheader()
            for row in rows:
                writer.writerow({k: row.get(k, "") for k in fieldnames})

    write_csv(output_dir / "nodes.csv", graph.get("nodes", []), ["id", "title", "subscribers"])
    write_csv(output_dir / "edges.csv", graph.get("edges", []), ["from", "to", "message_id"])

    message_rows = []
    for msg in graph.get("messages", []):
        msg_copy = dict(msg)
        msg_copy["reaction_breakdown"] = json.dumps(msg_copy.get("reaction_breakdown", {}), ensure_ascii=False)
        message_rows.append(msg_copy)

    write_csv(
        output_dir / "message_nodes.csv",
        message_rows,
        ["id", "chat", "message_id", "date", "url", "views", "reaction_count", "reaction_breakdown", "text", "sender_id"],
    )
    write_csv(
        output_dir / "message_edges.csv",
        graph.get("message_edges", []),
        ["from", "to", "type"],
    )

    summary = f"{len(graph.get('nodes', []))} nodes, {len(graph.get('messages', []))} messages"
    print(f"[export] {seed or output_dir.name}: {summary} -> {output_dir}")
    return output_dir


async def runCrawl(config: Dict) -> Dict[str, Dict[str, List[Dict[str, str]]]]:
    client = TelegramClient(config["sessionName"], int(config["apiId"]), str(config["apiHash"]))
    await client.start()
    print("Client connected")
    seeds = [seed for seed in config["startSeeds"] if seed]
    if not seeds:
        raise RuntimeError("No startSeed configured")

    graphs: Dict[str, Dict[str, List[Dict[str, str]]]] = {}
    try:
        for seed in seeds:
            graph = await crawlGraph(
                client,
                [seed],
                int(config["maxDepth"]),  # type: ignore[arg-type]
                config["maxPerChat"],
                float(config["handleDelay"]),  # type: ignore[arg-type]
                int(config["maxWait"]),  # type: ignore[arg-type]
                int(config["maxSubsForMessages"]),  # type: ignore[arg-type]
            )
            slug = normalize_seed(seed)
            graphs[slug] = graph
            output_dir = pathlib.Path(config["outputDir"]) / slug
            write_graph_outputs(graph, output_dir, slug)
    finally:
        await client.disconnect()
    return graphs


### Run crawl
Handles nest_asyncio if a loop is already running.


In [ ]:
import asyncio
try:
    asyncio.get_running_loop()
except RuntimeError:
    graphs = asyncio.run(runCrawl(config))
else:
    import nest_asyncio
    nest_asyncio.apply()
    graphs = await runCrawl(config)


Client connected
[skip] generationidentitaire: flood wait 16160s exceeds maxWait=600s; skipping
[export] generationidentitaire: 0 nodes, 0 messages -> data/generationidentitaire


### Export CSVs
Reads data/graph.json and writes CSVs next to it.


In [ ]:
import json, pathlib, csv

base_dir = pathlib.Path(config.get("outputDir", "data"))
if not base_dir.exists():
    raise FileNotFoundError(f'{base_dir} not found; run the crawler first.')

graph_paths = list(base_dir.glob("*/graph.json"))
if (base_dir / "graph.json").exists():
    graph_paths.append(base_dir / "graph.json")

for graph_path in graph_paths:
    graph = json.loads(graph_path.read_text(encoding='utf-8'))
    write_graph_outputs(graph, graph_path.parent, graph_path.parent.name)


[export] jungenationalisten: 120 nodes, 30254 messages -> data/jungenationalisten
[export] generationidentitaire: 0 nodes, 0 messages -> data/generationidentitaire
[export] afdjugendbw: 133 nodes, 31704 messages -> data/afdjugendbw
[export] tricoloredelsangueitalico: 158 nodes, 35936 messages -> data/tricoloredelsangueitalico
[export] data: 120 nodes, 30254 messages -> data
